# M3 · Tool de Normalización Terminológica (SNOMED CT vía BioPortal)


1. **Integración de herramienta externa** (SNOMED CT vía BioPortal).
2. **Criterio explícito de invocación** (se llama sólo si el score del retrieval de la entidad cruda es insuficiente).
3. **Manejo explícito de fallos** (campo `normalization_failed: True`, nunca fallback silencioso).


## 1 · Montar Google Drive (Datos y Outputs)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# PROJECT_ROOT para persistencia de datos
PROJECT_ROOT = Path('/content/drive/MyDrive/TopicosIA/Proyecto-Salud')
print('PROJECT_ROOT en Drive:', PROJECT_ROOT)

## 2 · Clonar el Repositorio 
Descarga el código actualizado directamente desde GitHub en el entorno de Colab.

In [ ]:
# Asegurarse de tener la version mas reciente de la rama luis
!rm -rf /content/Topicos-IA
!git clone --branch luis --single-branch https://github.com/luisNP21/Topicos-IA.git /content/Topicos-IA

REPO_DIR = Path('/content/Topicos-IA')
%cd {REPO_DIR}/M3/tools

## 3 · Instalación de Dependencias

In [ ]:
%pip install -q -r requirements.txt
print('Dependencias de Tools instaladas.')

## 4 · Ejecución de la Tool de Normalización CLI
Se evalúan las consultas de prueba. Si no se cuenta con `BIOPORTAL_API_KEY`, corre automáticamente en `MOCK_MODE` sin fallar.

In [ ]:
!python run_tools.py --config config.yaml

## 5 · Demostración del Criterio de Invocación y Manejo de Fallos
Verificamos la regla de orquestación ante score alto, score bajo con acierto, y fallo en la ontología.

In [ ]:
import sys
sys.path.insert(0, '/content/Topicos-IA/M3/tools')
from orquestacion import resolver_query, _retrieve_mock_bueno, _retrieve_mock_malo
from tool_normalizacion import normalizar_entidad

print('--- CASO 1: Score de retrieval alto (>= 0.5) -> Tool NO se invoca ---')
r1 = resolver_query('diabetes tipo 2', _retrieve_mock_bueno, umbral=0.5)
print(r1)

print('\n--- CASO 2: Score de retrieval bajo (< 0.5) -> Tool SÍ se invoca y normaliza ---')
r2 = resolver_query('diabetes tipo 2', _retrieve_mock_malo, umbral=0.5)
print(r2)

print('\n--- CASO 3: Falla de normalización -> Fallo explícito (normalization_failed: True) ---')
r3 = normalizar_entidad('enfermedad_inexistente_xyz_123')
print(r3)
